# 🎙️ VoiceLab: voice cloning with XTTS-v2 (Google Colab)

This notebook turns **a short recording of a voice + some text** into **new speech in a similar voice**.
It uses **XTTS-v2**, a pretrained model, so nothing is trained here. Colab gives you a free GPU, which makes it fast.

> **Status:** first version. It has not yet been run end to end by the author of this project.
> If a step fails, copy the red error text and ask for help. Errors are normal with fresh setups.

## Before you start

1. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Have your **cleaned reference file** ready. In VoiceLab, upload your recording and click **Download cleaned reference**. It is called `reference_clean.wav`.
3. Only use **your own voice** or the voice of someone who **gave you permission**.
4. Run the cells **one at a time, from top to bottom**. Click the ▶ button on the left of each cell and wait for it to finish (a green tick or the ▶ button returning).

Please **do not use "Run all"**, because some steps ask you to upload a file or change a setting.

## Step 1: Check that you have a GPU

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0), " <- good, you are ready")
else:
    print("NO GPU FOUND.")
    print("Go to Runtime -> Change runtime type, choose T4 GPU, Save, then run this cell again.")

## Step 2: Install the voice-cloning library (about 2-5 minutes)

We install **`coqui-tts`**, the maintained version of the Coqui TTS library that contains XTTS-v2.
Yellow warnings are normal. Only worry if the cell ends with a red error.

> If Colab shows a **"Restart session"** button or message, click it. After the restart,
> **skip this step** and continue from Step 3.

In [ ]:
import subprocess
import sys
import torch


def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


pip_install("coqui-tts")

# Newer PyTorch versions (2.9+) also need torchcodec, according to the coqui-tts instructions.
major, minor = (int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) >= (2, 9):
    pip_install("torchcodec")

print("Installation finished.")

## Step 3: Accept the conditions and load the model

XTTS-v2 is released under the **Coqui Public Model License (CPML)**, which is reported to allow **non-commercial use only**.
That is fine for a college project, but please read the license on the model page
(<https://huggingface.co/coqui/XTTS-v2>) yourself before you continue.

Change both settings below from `False` to `True` **only if they are true for you**, then run the cell.
The first run downloads the model (roughly 2 GB), so it takes a few minutes. Later runs in the same session are quick.

In [ ]:
I_HAVE_READ_AND_ACCEPT_THE_MODEL_LICENSE = False    # change to True after reading the license
I_HAVE_PERMISSION_TO_USE_THIS_VOICE = False         # change to True: it is my voice, or the owner agreed

if not (I_HAVE_READ_AND_ACCEPT_THE_MODEL_LICENSE and I_HAVE_PERMISSION_TO_USE_THIS_VOICE):
    raise RuntimeError("Please set both settings above to True (only if they are true), then run this cell again.")

import os
os.environ["COQUI_TOS_AGREED"] = "1"   # tells the library you accepted the license

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading XTTS-v2 on:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded.")

## Step 4: Upload your reference voice

Click **Choose files** and select `reference_clean.wav` (the file downloaded from VoiceLab).

Tips for a good clone:
- 10-30 seconds of **clear speech**, one speaker only
- no background music, no echo, no noise
- a normal speaking voice, not whispering or shouting

In [ ]:
from google.colab import files
from IPython.display import Audio, display
from scipy.io import wavfile

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded. Run this cell again and choose your WAV file.")

REFERENCE_WAV = list(uploaded.keys())[0]
try:
    sample_rate, data = wavfile.read(REFERENCE_WAV)
except Exception:
    raise RuntimeError("Could not read that file as WAV. Please upload the WAV downloaded from VoiceLab.")

seconds = len(data) / sample_rate
print(f"Loaded {REFERENCE_WAV}: {seconds:.1f} seconds at {sample_rate} Hz")
if seconds < 6:
    print("Warning: shorter than 6 seconds. The clone may sound weak. 10-30 seconds is better.")
display(Audio(REFERENCE_WAV))

## Step 5: Choose the text and language

Edit `TEXT` to whatever you want the cloned voice to say. Short sentences work best.
Set `LANGUAGE` to the language of the text. XTTS-v2 supports:
`en`, `es`, `fr`, `de`, `it`, `pt`, `pl`, `tr`, `ru`, `nl`, `cs`, `ar`, `zh-cn`, `ja`, `hu`, `ko`, `hi`.

In [ ]:
TEXT = "Hello! This is a test of my cloned voice, generated with VoiceLab."
LANGUAGE = "en"

if not TEXT.strip():
    raise ValueError("TEXT is empty. Type a sentence between the quotation marks.")
if len(TEXT) > 1000:
    print("Warning: that is a long text. Try a few sentences at a time for better quality.")
print("Text:", TEXT)
print("Language:", LANGUAGE)

## Step 6: Generate the speech

This usually takes a few seconds on the GPU. You can run this cell again after changing `TEXT`.

In [ ]:
import time
from IPython.display import Audio, display
from scipy.io import wavfile

start = time.time()
tts.tts_to_file(
    text=TEXT,
    speaker_wav=REFERENCE_WAV,
    language=LANGUAGE,
    file_path="generated.wav",
)
print(f"Done in {time.time() - start:.1f} seconds.")

out_rate, out_data = wavfile.read("generated.wav")
print(f"generated.wav: {len(out_data) / out_rate:.1f} seconds at {out_rate} Hz")
display(Audio("generated.wav"))

## Step 7: Download the result

Listen to the audio above first. Then download it and upload it in VoiceLab:
**SIP Analysis tab → Generated speech**, to see the waveform, spectrogram, pitch, MFCCs and the comparison.

In [ ]:
from google.colab import files
files.download("generated.wav")

## If something goes wrong

| Problem | What to do |
|---|---|
| "NO GPU FOUND" | Runtime → Change runtime type → T4 GPU → Save, then run Step 1 again |
| Colab asks to **restart the session** after Step 2 | Click Restart, then continue from Step 3 |
| Step 3 stops with "Please set both settings..." | Change the two `False` values to `True` (only if true for you) |
| Error mentioning memory / CUDA out of memory | Use shorter text; **Runtime → Restart session** and start again from Step 3 |
| The session was disconnected | Colab sessions expire. Re-run from Step 1 (Step 2 again if the library is gone) |
| The voice does not sound like me | Use a cleaner, longer (10-30 s) reference without noise or music |
| Any other red error | Copy the last few lines of the error and ask for help |

*Output sample rate and exact timings depend on the model version.*